# Day 8 - LLM foundations & API integration practical tasks
***

These tasks cover calling OpenAI API's and prompting.

## Explore how different model parameters and different prompts influence the response.
#### Here are some guidelines and advice you should consider:
* You can add as many cells as you need for each task.
* Use Markdown cells for your conclusions.
* For tasks that do not specify a model, you may choose any of the two available models: `gpt-5.6-luna` / `gpt-5.4-mini`.
* Within each comparison task, change only the model, prompt, or parameter specified in the instructions. Keep everything else - including the model, prompt, and other settings - the same across calls so you can compare the results fairly. For example, when testing different `temperature` values, use the same model, prompt, and output token limit (`max_output_tokens` for the Responses API or `max_completion_tokens` for the Chat Completions API).
* You can use both - [Responses API and Chat Completions API](https://medium.com/@sanathgoutham/openai-responses-api-vs-chat-completions-what-actually-changed-eeee86aff216) - to complete these tasks, but some model parameters are not supported by Responses API.
* LLMs understand [Markdown](https://www.markdownguide.org/cheat-sheet/) formatting in prompts. For example, wrapping text in double asterisks `(**important text**)` marks it as **bold** and can help signal emphasis.
* To copy text from a Markdown cell, open it in edit mode by clicking the pencil icon or double-clicking the cell.
* Optionally, paste API responses into a Markdown cell for better readability.
* Be creative and experiment.

In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ.pop("OPENAI_API_KEY", None)
load_dotenv()

True

In [7]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in the W2D3 .env file before running this cell.")

client = OpenAI(base_url="https://ds-ai-internship.openai.azure.com/openai/v1/",
                api_key=OPENAI_API_KEY)

#### 1. Write a Python script to call the OpenAI API and generate a response to the prompt below.

Prompt: `"What is artificial intelligence?"`

Use model `gpt-5.6-luna`.

In [8]:

from typing import Any

def call_response(prompt: str, model: str = "gpt-5.6-luna", **kwargs: Any):
    if not OPENAI_API_KEY:
        raise RuntimeError("Set OPENAI_API_KEY before calling the API.")
    response = client.responses.create(model=model, input=prompt, **kwargs)
    return response

def call_chat(prompt: str, model: str = "gpt-5.4-mini", **kwargs: Any):
    if not OPENAI_API_KEY:
        raise RuntimeError("Set OPENAI_API_KEY before calling the API.")
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        **kwargs,
    )
    return response

def display_response(response: Any):
    if hasattr(response, "output_text"):
        text = response.output_text
    elif hasattr(response, "choices") and response.choices:
        text = response.choices[0].message.content
    else:
        text = str(response)
    print(text.strip())

response = call_response(
    "What is artificial intelligence?",
    model="gpt-5.6-luna",
)
print(response.output_text)


Artificial intelligence (AI) is the field of creating computer systems that can perform tasks usually associated with human intelligence, such as:

- Understanding language
- Recognizing images or speech
- Learning from data
- Reasoning and solving problems
- Making predictions or decisions
- Generating text, images, music, or code

AI systems do not necessarily think or understand exactly like humans. They use algorithms and data to identify patterns and produce outputs. Examples include voice assistants, recommendation systems, self-driving features, fraud detection, and chatbots.


#### 2. Call the OpenAI API with different models

Prompt `"Explain the concept of machine learning to a 10-year-old."`

`model`: `gpt-5.6-luna` and `gpt-5.4-mini`.

Compare the outputs from each model. 

You can evaluate the differences in:
* Clarity of explanation.
* Simplicity of language (suitable for a 10-year-old).
* Creativity in the response.
* Response time.

In [18]:

prompt = "Explain the concept of machine learning to a 10-year-old."

for model in ["gpt-5.6-luna", "gpt-5.4-mini"]:
    print(f"=== {model} ===")
    response = call_response(prompt, model=model)
    print(response.output_text.strip())


=== gpt-5.6-luna ===
Machine learning is a way of teaching computers to learn from examples instead of giving them every instruction step by step.

Imagine showing a computer lots of pictures of cats and dogs:

- “This is a cat.”
- “This is a dog.”
- “This is a cat.”

After seeing many examples, the computer notices patterns—like ears, shapes, and fur. Later, when it sees a new picture, it can make a good guess about whether it shows a cat or a dog.

It’s similar to how you learn:

- You see many examples.
- You notice patterns.
- You use what you learned to make predictions.

Machine learning is used for things like:

- Recommending videos or songs
- Recognizing faces or voices
- Translating languages
- Helping cars notice roads and objects
- Filtering spam emails

But computers don’t understand things exactly like people do. They find patterns in data, and they can sometimes make mistakes—especially if their examples were limited or unfair.
=== gpt-5.4-mini ===
Machine learning is a 

#### 3. Call the OpenAI API with different `temperature` values

Prompt: `"Write a two sentence story about a robot learning to cook."`

`temperature` values: `0`, `0.7` and `1.5`. Generate three responses at each temperature.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare the outputs.

Tip: With Chat Completions API you can use parameter `n=3` to generate three different responses to your prompt with one call.

In [16]:

prompt = "Write a two sentence story about a robot learning to cook."

for temperature in [0, 0.7, 1.5]:
    print(f"=== temperature={temperature} ===")
    response = call_chat(
        prompt,
        model="gpt-5.4-mini",
        temperature=temperature,
        n=3,
        reasoning_effort="none",
    )
    for i, choice in enumerate(response.choices, start=1):
        print(f"Response {i}:{choice.message.content.strip()}")


=== temperature=0 ===
Response 1:The robot carefully studied recipes, its metal fingers clinking as it chopped vegetables and stirred a bubbling pot with precise, mechanical grace. When it finally served dinner, the robot’s sensors glowed with pride as everyone agreed the soup tasted surprisingly wonderful.
Response 2:The robot carefully studied recipes, measuring flour and stirring soup with precise, clanking motions until the kitchen smelled warm and delicious. When it finally served its first meal, it beeped with pride as its human friend smiled and asked for seconds.
Response 3:The robot carefully studied recipes, measuring flour and stirring soup with precise, clanking motions until the kitchen smelled warm and delicious. When it finally served its first meal, it beeped with pride as its human friend smiled and asked for seconds.
=== temperature=0.7 ===
Response 1:The robot carefully stirred the soup, consulting its recipe screen with the solemn focus of a scientist discovering fi

#### 4. Call the OpenAI API with different `max_output_tokens`/`max_completion_tokens` values.

Prompt: `"Explain the concept of machine learning in simple terms."`

`max_output_tokens` (Response API call)/`max_completion_tokens` (Chat Completions API call) values: `50`, `100` and `200`.

Use `gpt-5.4-mini` for this task.

Compare the length and detail of the responses.

In [17]:

prompt = "Explain the concept of machine learning in simple terms."

for max_tokens in [50, 100, 200]:
    print(f"=== max_output_tokens={max_tokens} ===")
    response = call_response(
        prompt,
        model="gpt-5.4-mini",
        max_output_tokens=max_tokens,
    )
    print(response.output_text.strip())


=== max_output_tokens=50 ===
Machine learning is a way for computers to learn from data instead of being told every rule step by step.

### Simple idea:
Imagine teaching a child to recognize cats. You show many pictures of cats and dogs. Over time, the child
=== max_output_tokens=100 ===
Machine learning is a way for computers to **learn from examples** instead of being told every single rule by hand.

### Simple idea
Imagine teaching a child to recognize cats:
- You show many pictures of cats and dogs.
- Over time, the child starts noticing patterns, like ears, fur, and whiskers.
- Then they can पहचानना recognize a new cat they’ve never seen before.

Machine learning works similarly:
- You give a computer lots of data.
- It looks
=== max_output_tokens=200 ===
Machine learning is a way for computers to **learn patterns from data** instead of being explicitly programmed for every rule.

### Simple idea:
Instead of telling a computer:

- “If this email has these words, mark it as spam”
-

#### 5. Call the OpenAI API with different `top_p` values.

Prompt: `"Write a list of 5 completely random and unusual hobbies that no one has ever heard of."`

`top_p` values: `0.3`, `0.7`, and `1.0`.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare response variation as you adjust `top_p`.

In [19]:

prompt = "Write a list of 5 completely random and unusual hobbies that no one has ever heard of."

for top_p in [0.3, 0.7, 1.0]:
    print(f"=== top_p={top_p} ===")
    response = call_chat(
        prompt,
        model="gpt-5.4-mini",
        top_p=top_p,
        reasoning_effort="none",
    )
    print(response.choices[0].message.content.strip())


=== top_p=0.3 ===
Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow spotting** — tracking and sketching the shapes clouds cast on the ground at different times of day.  
2. **Vintage elevator appreciation** — collecting stories, photos, and details about old elevators and their designs.  
3. **Pebble balancing in rain** — building tiny stone towers specifically during wet weather to see how water changes stability.  
4. **Forgotten menu archiving** — hunting down old restaurant menus and cataloging bizarre dishes that no longer exist.  
5. **Echo mapping** — visiting unusual spaces like tunnels, stairwells, and empty halls to record how sound bounces differently in each one.
=== top_p=0.7 ===
Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow spotting** — tracking and naming the shapes that clouds cast on the ground at different times of day.  
2. **Button archaeology** — collecting old buttons and inventing fictional histories for where the

#### 6. Call the OpenAI API with different `frequency_penalty` values.

Prompt: `"Write a list of funny one-liner jokes about cats."`

`frequency_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the repetition of joke formats changes as you increase the `frequency_penalty`.

In [20]:

prompt = "Write a list of funny one-liner jokes about cats."

for penalty in [0, 0.5, 1.0]:
    print(f"=== frequency_penalty={penalty} ===")
    response = call_chat(
        prompt,
        model="gpt-5.4-mini",
        frequency_penalty=penalty,
        reasoning_effort="none",
    )
    print(response.choices[0].message.content.strip())


=== frequency_penalty=0 ===
Sure—here are some funny cat one-liners:

- Cats are like potato chips: you can’t have just one, and they’ll judge you for trying.
- My cat doesn’t do tricks—he does emotional manipulation.
- I asked my cat for help, and he just stared at me like I was the problem.
- A cat’s favorite workout is pouncing on your dignity at 3 a.m.
- My cat thinks the litter box is a luxury spa.
- Cats: the only pets who act like they pay rent.
- I bought my cat a bed, and he still sleeps in my clean laundry out of spite.
- My cat doesn’t meow—he makes tiny demands.
- They say curiosity killed the cat, but in my house it mostly breaks a vase.
- My cat’s favorite hobby is knocking things off tables to keep me humble.
- If cats could text, they’d only send “?” and judgment.
- My cat is basically a furry roommate with no job and strong opinions.
- I tried to teach my cat to fetch, but he just looked offended and left.
- Cats don’t have owners; they have staff.
- My cat’s idea of “

#### 7. Call the OpenAI API with different `presence_penalty` values.

Prompt: `"Suggest ten names for a new line of eco-friendly cleaning products."`

`presence_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the diversity of generated names changes as you increase the `presence_penalty`.

In [21]:

prompt = "Suggest ten names for a new line of eco-friendly cleaning products."

for penalty in [0, 0.5, 1.0]:
    print(f"=== presence_penalty={penalty} ===")
    response = call_chat(
        prompt,
        model="gpt-5.4-mini",
        presence_penalty=penalty,
        reasoning_effort="none",
    )
    print(response.choices[0].message.content.strip())


=== presence_penalty=0 ===
Here are ten name ideas for an eco-friendly cleaning product line:

1. **Green Gleam**
2. **PureNest**
3. **EcoShine**
4. **FreshCycle**
5. **CleanEarth**
6. **Leaf & Lather**
7. **NatureSpark**
8. **BioBright**
9. **KindClean**
10. **Verdant Vue**

If you want, I can also generate names with a more **luxury**, **playful**, **minimalist**, or **science-forward** vibe.
=== presence_penalty=0.5 ===
Here are ten name ideas for an eco-friendly cleaning product line:

1. **Green Gleam**
2. **EcoSpark**
3. **PurePlanet Clean**
4. **FreshRoot**
5. **EarthShine**
6. **Verdant убор?**
=== presence_penalty=1.0 ===
Here are ten name ideas for an eco-friendly cleaning product line:

1. **GreenGlow**
2. **PurePlanet**
3. **EcoShine**
4. **FreshTerra**
5. **CleanLeaf**
6. **EarthSpark**
7. **Verdant Clean**
8. **NatureNest**
9. **BloomBright**
10. **TrueEco Clean**

If you want, I can also suggest names with a more **premium**, **playful**, or **scientific/modern** feel.


#### 8. Use the prompt "*Explain the importance of data preprocessing in machine learning.*" with the following system prompts:
1. "You are a university professor teaching a class of beginners."
2. "You are a technical expert explaining to a team of engineers."
3. "You are a comedian explaining this concept in a funny way."

Observe how the tone and style of the response change based on the system prompt.

In [22]:

user_prompt = "Explain the importance of data preprocessing in machine learning."
system_prompts = [
    "You are a university professor teaching a class of beginners.",
    "You are a technical expert explaining to a team of engineers.",
    "You are a comedian explaining this concept in a funny way.",
]

for system_prompt in system_prompts:
    print(f"=== system prompt: {system_prompt} ===")
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        reasoning_effort="none",
    )
    print(response.choices[0].message.content.strip())


=== system prompt: You are a university professor teaching a class of beginners. ===
Data preprocessing is very important in machine learning because raw data is usually messy, incomplete, and inconsistent. Good preprocessing helps models learn better and gives more reliable results.

### Why it matters

1. **Improves data quality**  
   Real-world data often has missing values, duplicates, errors, or noise. Preprocessing cleans these issues so the model does not learn from bad data.

2. **Makes features usable by algorithms**  
   Many machine learning algorithms work better when data is in a certain form. For example, numerical features may need to be scaled, and categorical values may need to be encoded as numbers.

3. **Helps models train more effectively**  
   If features have very different scales, some algorithms may struggle. Preprocessing like normalization or standardization can make training faster and more stable.

4. **Increases accuracy**  
   Better-prepared data usuall

#### 9. Write a Zero-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!".

Use it in your API call.

In [23]:

review = "The product is amazing, and I love using it every day!"
zero_shot_prompt = (
    "Classify the sentiment of the following product review as positive, neutral, or negative. "
    f"Review: {review}"
)

response = call_chat(zero_shot_prompt, model="gpt-5.4-mini", reasoning_effort="none")
print(response.choices[0].message.content.strip())


positive


#### 10. Write a Few-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!"

Use it in your API call.

Observe how the model's performance changes and compare the response to Zero-Shot output.

In [25]:
review = "The product is amazing, and I love using it every day!"

few_shot_prompt = (
    "Classify the sentiment of the review as positive, neutral, or negative.\n\n"
    "Examples:\n"
    '- Review: "The packaging was damaged and the product did not work." -> negative\n'
    '- Review: "The product is okay, but I expected more." -> neutral\n'
    '- Review: "The product is amazing, and I love using it every day!" -> positive\n\n'
    f"Review: {review}\n"
)

response = call_chat(
    few_shot_prompt,
    model="gpt-5.4-mini",
    reasoning_effort="none"
)

print(response.choices[0].message.content.strip())

positive


#### 11. Optimize input tokens.

Prompt: `"As you know, streaming services and social media platforms use artificial intelligence to recommend movies, shows, and songs to people. These smart recommendation systems work by looking closely at your past viewing habits, what categories you like, and what similar users enjoy watching, and then they predict what you might want to see next. Keeping this background in mind, explain in two simple sentences why a user who loves sci-fi movies suddenly sees a romantic comedy recommended."`

Rewrite the prompt to use as few input tokens as possible while preserving its meaning and the requirement to answer in two simple sentences. The response should remain relevant, clear, and accurate.

Run the original and revised prompts using the same model and settings. Compare their input token counts and assess whether shortening the prompt affected the response quality.

You can use the API response’s usage information to check token counts.

In [26]:

original_prompt = (
    "As you know, streaming services and social media platforms use artificial intelligence to "
    "recommend movies, shows, and songs to people. These smart recommendation systems work by "
    "looking closely at your past viewing habits, what categories you like, and what similar users "
    "enjoy watching, and then they predict what you might want to see next. Keeping this "
    "background in mind, explain in two simple sentences why a user who loves sci-fi movies"
    " suddenly sees a romantic comedy recommended."
)
revised_prompt = (
    "A sci-fi fan suddenly gets a romantic comedy recommendation because recommendations use "
    "past viewing habits, similarity to other users, and category preferences to predict likely "
    "interests. Explain this in two simple sentences."
)

original_response = call_response(original_prompt, model="gpt-5.4-mini")
revised_response = call_response(revised_prompt, model="gpt-5.4-mini")

print(f"Original input tokens: {original_response.usage.input_tokens}")
print(f"Revised input tokens: {revised_response.usage.input_tokens}")
print("=== Original ===")
print(original_response.output_text.strip())
print("=== Revised ===")
print(revised_response.output_text.strip())


Original input tokens: 94
Revised input tokens: 43
=== Original ===
The recommendation system may suggest a romantic comedy because it noticed that users who like sci-fi also often enjoy romantic comedies. It can also happen because the system wants to show something different that it thinks the user might like based on other past choices or similar users.
=== Revised ===
The recommendation system looks at what the user has watched before, what similar users enjoyed, and the genres they usually like. Based on those patterns, it predicts that a romantic comedy might also interest them.


#### 12. Optimize output tokens.

Prompt: `"Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"`

Rewrite the prompt to request a concise, plausible explanation in 16 output tokens or fewer. This respects the API minimum while still testing a shorter response.

Run the original and revised prompts using the same model and settings. Compare their output token counts and check whether the shorter response still provides a meaningful explanation. Refine your prompt if necessary.

You can use the API response?s usage information to check token counts.

In [28]:
original_prompt = "Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"
concise_prompt = "Why did my app suggest a romantic comedy when I mostly watch sci-fi? Reply in 16 output tokens or fewer."

original_response = call_response(original_prompt, model="gpt-5.4-mini", max_output_tokens=30)
concise_response = call_response(concise_prompt, model="gpt-5.4-mini", max_output_tokens=16)

print(f"Original output tokens: {original_response.usage.output_tokens}")
print(f"Concise output tokens: {concise_response.usage.output_tokens}")
print("=== Original ===")
print(original_response.output_text.strip())
print("=== Concise ===")
print(concise_response.output_text.strip())


Original output tokens: 30
Concise output tokens: 16
=== Original ===
A few common reasons:

- **You watched something adjacent**: If you recently watched a movie with romance or comedy elements, the app
=== Concise ===
Algorithms test patterns, not just favorites, and moods shift.


#### 13. Expand the given prompt accordingly to achieve the requested output from the list below.

Prompt: `"Explain the importance of exercise." `

Expected output (from three separate prompts):
1. One sentence long
2. Paragraph format
3. Bullet points

In [29]:

prompt_one_sentence = "Explain the importance of exercise in one sentence."
prompt_paragraph = "Explain the importance of exercise in a concise paragraph."
prompt_bullets = "Explain the importance of exercise in bullet points with 4-5 items."

for label, prompt in [
    ("One sentence", prompt_one_sentence),
    ("Paragraph", prompt_paragraph),
    ("Bullet points", prompt_bullets),
]:
    print(f"=== {label} ===")
    response = call_response(prompt, model="gpt-5.4-mini")
    print(response.output_text.strip())


=== One sentence ===
Exercise is important because it improves physical health, boosts mental well-being, and helps maintain energy and mobility throughout life.
=== Paragraph ===
Exercise is important because it strengthens the heart, muscles, and bones while improving overall fitness and energy levels. It also helps maintain a healthy weight, lowers the risk of many chronic diseases, and supports better mental health by reducing stress and boosting mood. Regular physical activity can improve sleep, focus, and quality of life, making it an essential part of a healthy lifestyle.
=== Bullet points ===
- **Improves physical health:** Exercise strengthens the heart, muscles, and bones, and helps reduce the risk of diseases like obesity, diabetes, and high blood pressure.  
- **Boosts mental health:** It can reduce stress, anxiety, and depression while improving mood and overall emotional well-being.  
- **Increases energy and stamina:** Regular activity helps the body work more efficientl

#### 14. Expand this prompt to get more creative responses.

Prompt: `"Explain Artificial Intelligence."`

Expected output (from three separate prompts):
1. Metaphor
2. Story
3. Funny

In [46]:

prompts = {
    "Metaphor": "Explain artificial intelligence using a metaphor that compares it to a familiar everyday task.",
    "Story": "Explain artificial intelligence as a short story with a clear beginning, middle, and end.",
    "Funny": "Explain artificial intelligence in a funny, memorable way suitable for a general audience.",
}

for label, prompt in prompts.items():
    print(f"=== {label} ===")
    response = call_response(prompt, model="gpt-5.4-mini")
    print(response.output_text.strip())


=== Metaphor ===
Artificial intelligence is like **learning to cook by watching thousands of recipes and tasting many dishes**.

At first, you don’t know much. But over time, you notice patterns:

- which ingredients usually go together,
- how much salt improves flavor,
- how long something needs to bake.

AI works similarly. It studies lots of examples, finds patterns, and then uses those patterns to make predictions or decisions on new things it hasn’t seen before.

So if a person learns to cook by experience, AI “learns” by analyzing data. It doesn’t understand food the way a human does, but it can become very good at recognizing what tends to work.
=== Story ===
Once upon a time, in a small town filled with busy people, there was a young inventor named Mira. She wanted help organizing her shop, remembering customers, and predicting what supplies she would need. So she built a clever machine that could learn from examples. She called it “AI.”

At first, AI was like a curious child. 

#### 15. Expand this prompt for different audiences.

Prompt: `"Explain climate change."`

Expected audience (from three separate prompts):
1. Kids
2. General public
3. Politicians

In [31]:

prompts = {
    "Kids": "Explain climate change to a child using simple language and a cheerful tone.",
    "General public": "Explain climate change to the general public in a balanced and informative way.",
    "Politicians": "Explain climate change to politicians with a practical, policy-focused framing.",
}

for label, prompt in prompts.items():
    print(f"=== {label} ===")
    response = call_response(prompt, model="gpt-5.4-mini")
    print(response.output_text.strip())


=== Kids ===
Sure! Here’s a kid-friendly explanation:

Climate change means the Earth is getting warmer little by little.  
The Earth has a blanket around it made of gases, and that blanket helps keep heat in. But when people use lots of cars, factories, and electricity made from coal, oil, and gas, that blanket gets too thick and traps too much heat.

This can make:
- summers hotter
- storms stronger
- ice at the North and South Poles melt
- some animals have a harder time finding homes

But the good news is: people can help! We can:
- save energy
- walk, bike, or use public transport
- recycle and reuse things
- plant trees
- turn off lights when we don’t need them

So climate change is a big problem, but everyone can help make the Earth healthier and happier!
=== General public ===
Climate change refers to long-term shifts in Earth’s climate, especially the warming trend observed since the late 1800s. The key issue today is not simply that climate varies naturally—it always has—but 

#### 16. Expand this prompt accordingly to instructions below.

Prompt: `"Describe a sunset."`

Expected result (from three separate prompts):

1. Add more details to the prompt, e.g., on what details should it focus on.
2. Constrain the prompt by forbidding it to use words 'sun' and 'light'.
3. Change the output in the style of Shakespeare.

In [32]:

prompts = {
    "Detailed": "Describe a sunset in vivid detail, focusing on the changing colors, clouds, and feelings it creates.",
    "No sun/light words": "Describe a sunset without using the words 'sun' or 'light'. Focus on color, mood, and atmosphere.",
    "Shakespeare": "Describe a sunset in the style of Shakespeare, with poetic imagery and dramatic language.",
}

for label, prompt in prompts.items():
    print(f"=== {label} ===")
    response = call_response(prompt, model="gpt-5.4-mini")
    print(response.output_text.strip())


=== Detailed ===
The sunset begins softly, as if the sky is holding its breath. At first there’s a pale wash of gold near the horizon, thin and delicate, like light poured through honey. Then the colors deepen. Amber gives way to apricot, apricot to coral, and the coral burns brighter until it glows with a rich, molten red. Above it all, the blue of day slowly cools and thins into lavender, then indigo, as though night is quietly stepping in behind the fading light.

Clouds catch the fire as it spreads. A long, feathered streak becomes edged in rose and tangerine, its underside lit like embers in a dying hearth. Puffy clouds drift across the sky in soft layers, their tops brushed with pearl and peach while their bellies darken into mauve and slate. Some clouds look like they’re smoldering from within; others seem almost translucent, glowing at the edges as if the sun has turned them into stained glass. Where the light strikes thin cloud, it flashes briefly with a blaze of gold so brigh

#### 17. Write your observations from tasks 13-16, e.g., how different prompts influence complexity, tone, style and focus of the responses.

Observations:
- More specific prompts produced clearer and more focused answers, while vague prompts led to broader or less useful responses.
- Changing the audience (e.g., child, general reader, or politician) noticeably changed the vocabulary, level of detail, and framing of the explanation.
- Restricting the output format or banning certain words forced the model to adapt its style and often made the response more creative or more constrained.
- Longer prompts with explicit tasks and constraints usually improved consistency, but they also increased the risk of over-explaining if the instructions were too open-ended.

Conclusion:
Prompt engineering is a major factor in the quality of LLM output. Small changes in wording, audience, and constraints can dramatically affect tone, structure, and usefulness, so good prompts should be precise, intentional, and aligned with the target outcome.


#### 18. Generate a Python code with API call.

Write a prompt that generates a Python function to calculate the factorial of a number. Once you have written the prompt, use it to call the API and generate the code. Test the generated code to ensure it calculates the factorial correctly for various inputs.

In [34]:

import re

prompt = (
    "Write a Python function that calculates the factorial of a number. "
    "Return only the Python code, with no markdown fences or explanations, and make sure the code is correct."
)
response = call_response(prompt, model="gpt-5.4-mini")
code = response.output_text.strip()
code = re.sub(r"^```(?:python)?\s*", "", code, flags=re.IGNORECASE)
code = re.sub(r"\s*```$", "", code)
print(code)

namespace = {}
exec(code, namespace)
factorial = namespace.get("factorial")
if factorial is None:
    raise RuntimeError("Generated code did not define a factorial function.")

for value in [0, 1, 5, 7]:
    print(f"factorial({value}) = {factorial(value)}")


def factorial(n):
    if not isinstance(n, int):
        raise TypeError("factorial() only accepts integers")
    if n < 0:
        raise ValueError("factorial() not defined for negative values")
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
factorial(0) = 1
factorial(1) = 1
factorial(5) = 120
factorial(7) = 5040


#### 19. Provide the following buggy Python code to the API and ask it to fix the errors:

```
def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)
```

Check if the fixed code is working.

In [36]:

import re

buggy_code = '''
def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)
'''
prompt = (
    "Fix the following Python code and return only the corrected Python code, with no markdown fences or explanations. "
    "Make sure the function definition and the print call are valid: \n\n"
    f"{buggy_code}"
)
response = call_response(prompt, model="gpt-5.4-mini")
fixed_code = response.output_text.strip()
fixed_code = re.sub(r"^```(?:python)?\s*", "", fixed_code, flags=re.IGNORECASE)
fixed_code = re.sub(r"\s*```$", "", fixed_code)
print(fixed_code)

namespace = {}
exec(fixed_code, namespace)
print("Execution succeeded.")


def add_numbers(a, b):
    return a + b

print(add_numbers(5, 10))
15
Execution succeeded.


#### 20. Summarize this text to 2-3 sentences by using API call.

"Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user’s prompt or request.

At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that’s similar, but not identical, to the original data.

Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:

Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.

Diffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.

Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today’s headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney."

[<p align="right"> <cite> Source: What is generative AI? </cite></p>](https://www.ibm.com/think/topics/generative-ai)

In [37]:

text = '''
Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user's prompt or request.

At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that's similar, but not identical, to the original data.

Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:

Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.

Diffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.

Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today's headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney.
'''
summary_prompt = "Summarize the text in 2-3 sentences while preserving the key ideas."
response = call_response(summary_prompt + "\n\nText:\n" + text, model="gpt-5.4-mini")
print(response.output_text.strip())


Generative AI, or gen AI, uses deep learning models to create new content such as text, images, video, and audio in response to prompts by learning simplified patterns from training data and generating similar but original outputs. Its major recent advances come from three model types: VAEs, diffusion models, and transformers, with transformers powering many leading tools like ChatGPT, GPT-4, Copilot, BERT, Bard, and Midjourney.


#### 21. Translate the following idioms into a language of your choice, ensuring the meaning is preserved (not just word-for-word translation):

1. "It's raining cats and dogs."
2. "Break a leg."
3. "Kill two birds with one stone."
4. "Spill the beans."
5. "A blessing in disguise."

Try to improve the response as much as possible and evaluate the accuracy of the final output.

In [38]:

    idioms = [
        "It's raining cats and dogs.",
        "Break a leg.",
        "Kill two birds with one stone.",
        "Spill the beans.",
        "A blessing in disguise.",
    ]
    prompt = (
        "Translate each idiom below into Spanish while preserving the intended meaning, not just the literal wording. "
        "Return the results as a numbered list.\n\n" + "\n".join(f"{i+1}. {item}" for i, item in enumerate(idioms))
    )
    response = call_response(prompt, model="gpt-5.4-mini")
    print(response.output_text.strip())


1. Está lloviendo a cántaros.  
2. ¡Mucha mierda!  
3. Matar dos pájaros de un tiro.  
4. Soltar la sopa.  
5. Una bendición disfrazada.


#### 22. Generate a paragraph about a cat without using the letter 'e'/'E'.

Write a prompt that forces the AI to produce a three-sentence story about a cat while strictly forbidding it to use letter 'e'.

Use `gpt-5.4-mini` for this task.

Explain why your prompt is effective.

In [39]:

prompt = (
    "Write a three-sentence story about a cat. Strictly forbid the letter 'e' or 'E' in the entire output. "
    "The story must still be coherent and readable."
)
response = call_response(prompt, model="gpt-5.4-mini")
print(response.output_text.strip())


A sly cat sat by a warm lamp, calm and spry.  
It chased a red ball, then napped on a soft rug.  
At dawn, it drank milk and purred with pure joy.


#### 23. Extract JSON Data from the input text in two different ways.

Input Text: "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."

Extract the information from the text above using two different methods:

1. Write a prompt asking the AI to return a JSON object with the fields `destination`, `start_date`, and `stay_duration`.

2. Instead of putting rules in the prompt, provide a specific JSON schema that defines the field names and their required data types `destination` (the first city), `start_date` (formatted as YYYY-MM-DD), and `stay_duration` (the number of nights as an integer).

Which method yields a more reliable result for a computer program to process, and why?

In [40]:

input_text = "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."
prompt = (
    "Extract the destination, start date, and stay duration from the following text and return a JSON object. "
    "Use keys: destination, start_date, stay_duration.\n\n" + input_text
)
response = call_response(prompt, model="gpt-5.4-mini")
print(response.output_text.strip())


{"destination":"Tokyo","start_date":"June 12th","stay_duration":"5 nights"}


In [41]:

from pydantic import BaseModel

class TripInfo(BaseModel):
    destination: str
    start_date: str
    stay_duration: int

schema_prompt = (
    "Extract the information from the following text and return valid JSON that matches the required schema. "
    "The destination should be the city name, the start_date must be YYYY-MM-DD, and stay_duration must be an integer number of nights.\n\n"
    "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."
)
response = call_response(schema_prompt, model="gpt-5.4-mini")
print(response.output_text.strip())


{"destination":"Tokyo","start_date":"2025-06-12","stay_duration":5}


#### 24. Create a structured table comparing different programming languages.

Write a prompt that generates a table in this format (example below) with accurate information.

| Language    | Use Case               | Advantages                     | Disadvantages                |
|-------------|------------------------|---------------------------------|------------------------------|
| Python      | Data Science, AI       | Easy to learn, versatile       | Slower execution speed       |
| Java        | Enterprise Applications| Platform-independent, robust   | Verbose syntax               |
| JavaScript  | Web Development        | Fast, widely supported         | Limited for backend tasks    |

**Tip**: To enhance readability, you can copy and paste the generated table into a Markdown cell.

In [42]:

prompt = (
    "Create a markdown table comparing Python, Java, and JavaScript. Include columns for Language, Use Case, Advantages, and Disadvantages. "
    "Use accurate, concise information."
)
response = call_response(prompt, model="gpt-5.4-mini")
print(response.output_text.strip())


| Language | Use Case | Advantages | Disadvantages |
|---|---|---|---|
| Python | Data science, automation, web development, scripting, AI/ML | Easy to read and learn, large ecosystem, rapid development | Slower than compiled languages, can be less suited for mobile/frontend, GIL can limit CPU-bound threading |
| Java | Enterprise applications, backend systems, Android development, large-scale software | Strong performance, portable via JVM, robust tooling, statically typed | Verbose syntax, slower startup, more complex than Python for small scripts |
| JavaScript | Web development, frontend UI, backend with Node.js | Runs in all browsers, essential for web apps, flexible and widely supported | Inconsistent behavior across environments, dynamic typing can cause bugs, can become hard to maintain at scale |


#### 25. Generate a quiz about machine learning basics.

Look at the example below and write a prompt that generates a quiz with identical formatting. Ensure the title formatting, list markers (a, b, c, d), and the correct answer indicator (*) are accurately replicated.

**Machine learning basics quiz**

1. Which of the following best describes supervised learning?  
a) The model learns from unlabeled data to find hidden patterns  
b) The model learns from labeled data to make predictions *  
c) The model is programmed with explicit rules  
d) The model randomly guesses outputs without training

2. Which of these is a common application of machine learning in everyday life?  
a) Sorting files manually on a computer  
b) Email spam filtering *  
c) Writing code by hand  
d) Using a calculator for arithmetic

3. What is overfitting in machine learning?  
a) When the model performs well on new, unseen data  
b) When the model is too simple and underperforms  
c) When the model learns the training data too well, including noise, reducing generalization *  
d) When the model runs faster than expected

4. Which ethical concern is commonly associated with machine learning algorithms?  
a) Increased hardware costs  
b) Bias in training data leading to unfair predictions *  
c) The inability to process large datasets  
d) Lack of available programming languages

5. Which of the following is NOT a type of machine learning?  
a) Supervised learning  
b) Reinforcement learning  
c) Unsupervised learning  
d) Manual learning *

In [45]:

prompt = '''
Machine learning basics quiz

1. Which of the following best describes supervised learning?
a) The model learns from unlabeled data to find hidden patterns
b) The model learns from labeled data to make predictions *
c) The model is programmed with explicit rules
d) The model randomly guesses outputs without training

2. Which of these is a common application of machine learning in everyday life?
a) Sorting files manually on a computer
b) Email spam filtering *
c) Writing code by hand
d) Using a calculator for arithmetic

3. What is overfitting in machine learning?
a) When the model performs well on new, unseen data
b) When the model is too simple and underperforms
c) When the model learns the training data too well, including noise, reducing generalization *
d) When the model runs faster than expected

4. Which ethical concern is commonly associated with machine learning algorithms?
a) Increased hardware costs
b) Bias in training data leading to unfair predictions *
c) The inability to process large datasets
d) Lack of available programming languages

5. Which of the following is NOT a type of machine learning?
a) Supervised learning
b) Reinforcement learning
c) Unsupervised learning
d) Database management *

Generate a similar quiz with five questions using the same format and answer marker style.
'''
response = call_response(prompt, model="gpt-5.4-mini")
print(response.output_text.strip())


Machine learning basics quiz

1. Which of the following best describes unsupervised learning?
a) The model learns from labeled data to make predictions
b) The model learns from unlabeled data to discover patterns *
c) The model follows fixed human-written rules
d) The model only works with images

2. Which task is a typical example of regression in machine learning?
a) Predicting whether an email is spam
b) Grouping customers into segments
c) Predicting house prices *
d) Recognizing handwritten digits

3. What is the main goal of a training dataset?
a) To evaluate the final model only
b) To help the model learn patterns from examples *
c) To store the model after deployment
d) To remove all errors from the data

4. Which of the following is a common method used to reduce overfitting?
a) Increasing model complexity
b) Using regularization *
c) Removing all training data
d) Avoiding validation tests

5. In reinforcement learning, what does an agent learn from?
a) Only labeled examples
b)

#### 26. Create a comprehensive online course outline for beginners.

Write a prompt that generates entire course outline, following the example provided below. Ensure the prompt is detailed enough to cover all sections, with clear formatting and practical guidance. 

Sections to include in course outline:
- Course overview - description, objectives, target audience.
- Module breakdown - 5 modules with title, short description, three key learning outcomes and an interactive assignment in the end of each module.
- Assessment plan - a table outlining the assessment methods with description and weighting.
- Certification criteria - requirements to earn a certificate.
- Marketing strategy - step-by-step guide for promoting the course.

Include a detailed description of the type of information each section should contain. Maintain the **same formatting**, **section titles**, and **course name**. While the generated descriptions, modules and marketing strategy steps may differ and incorporate new ideas and concepts, it must remain aligned with the theme of the task.

Put the final output of your API call in a Markdown cell. 

#### Example:
---
---
## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview
This beginner-friendly course is designed to equip learners with the foundational skills and knowledge required to craft effective prompts for AI applications. By mastering prompt engineering, students will learn how to optimize AI models for various tasks, including text generation, data analysis, and creative problem-solving. The course is ideal for individuals interested in AI, developers, data scientists, and anyone looking to leverage AI tools in their personal or professional projects.

#### Objectives
- Understand the principles of prompt engineering and its role in AI workflows.
- Learn techniques to design, test, and refine prompts for optimal AI performance.
- Apply prompt engineering skills to real-world scenarios and projects.

#### Target Audience
- Beginners with no prior experience in AI or programming.
- Professionals seeking to integrate AI into their workflows.
- Students and enthusiasts curious about AI capabilities.

---

### Module Breakdown

#### Module 1: Introduction to Prompt Engineering
- **Description:** This module introduces the concept of prompt engineering, its importance in AI applications, and the basics of interacting with AI models.
- **Key Learning Outcomes:**
  1. Define prompt engineering and its significance in AI workflows.
  2. Understand the structure and components of effective prompts.
  3. Explore real-world use cases of prompt engineering.
- **Interactive Assignment:** Write a simple prompt for a text-generation AI model to generate a short story.

---

#### Module 2: Principles of Effective Prompt Design
- **Description:** Learn the core principles and techniques for designing prompts that yield accurate and relevant AI outputs.
- **Key Learning Outcomes:**
  1. Identify factors that influence AI responses (e.g., context, specificity, tone).
  2. Explore strategies for iterative prompt refinement.
  3. Understand common pitfalls and how to avoid them.
- **Interactive Assignment:** Design and test three variations of a prompt for a chatbot to answer customer queries effectively.

---

#### Module 3: Advanced Techniques in Prompt Engineering
- **Description:** Dive deeper into advanced methods, including chaining prompts, dynamic prompts, and leveraging AI tools for complex tasks.
- **Key Learning Outcomes:**
  1. Learn how to chain prompts for multi-step tasks.
  2. Explore dynamic prompt techniques for adaptive AI responses.
  3. Understand how to integrate external data sources into prompts.
- **Interactive Assignment:** Create a multi-step prompt to generate a detailed report based on user-provided data.

---

#### Module 4: Applications of Prompt Engineering Across Industries
- **Description:** Discover how prompt engineering is applied in various industries, such as healthcare, education, marketing, and software development.
- **Key Learning Outcomes:**
  1. Analyze industry-specific use cases of prompt engineering.
  2. Learn how to tailor prompts for different domains and objectives.
  3. Explore ethical considerations in prompt engineering.
- **Interactive Assignment:** Develop a prompt for a marketing campaign, including product descriptions and social media posts.

---

#### Module 5: Building and Testing AI Projects with Prompt Engineering
- **Description:** Apply your skills to design, test, and deploy AI projects using prompt engineering techniques.
- **Key Learning Outcomes:**
  1. Plan and execute an AI project using prompt engineering.
  2. Test and refine prompts for optimal project outcomes.
  3. Document and present your project effectively.
- **Interactive Assignment:** Create a mini-project using prompt engineering to solve a real-world problem (e.g., automating customer support).

---

### Assessment Plan

| Assessment Method      | Description                          | Weightage (%) |
|------------------------|--------------------------------------|---------------|
| Quizzes               | Multiple-choice quizzes after each module | 30%          |
| Assignments           | Practical exercises for each module  | 40%           |
| Final Project         | A comprehensive project combining all skills | 20%          |
| Participation         | Active participation in discussions  | 10%           |

---

### Certification Criteria
To earn a certificate of completion, students must:
1. Achieve a minimum score of 70% across all assessments.
2. Submit all interactive assignments and the final project.
3. Actively participate in at least 80% of discussion forums and live sessions.

---

### Marketing Strategy

#### Step 1: Social Media Campaigns
- Create engaging posts highlighting the course benefits, modules, and outcomes.
- Share testimonials and success stories from beta testers or early participants.
- Use targeted ads on platforms like LinkedIn, Facebook, and Instagram.

#### Step 2: Email Marketing
- Build an email list of potential learners (e.g., professionals, students).
- Send a series of emails introducing the course, its benefits, and enrollment deadlines.
- Include exclusive early-bird discounts and referral incentives.

#### Step 3: Partnerships
- Collaborate with universities, coding bootcamps, and AI communities to promote the course.
- Partner with influencers and thought leaders in the AI space for endorsements.

#### Step 4: Content Creation
- Publish blog posts, videos, and webinars on topics related to prompt engineering.
- Share free resources (e.g., mini-guides, templates) to attract interest.
- Host live Q&A sessions to engage with the audience and answer questions about the course.

#### Step 5: Testimonials and Reviews
- Collect feedback from early participants and showcase their success stories.
- Encourage learners to share their experiences on social media and review platforms.

---

In [44]:

prompt = '''
Create a course outline called "Mastering Prompt Engineering for AI Applications".

Use the following structure exactly:
## Course Overview
### Module 1: ...
- Learning outcomes:
- Assignment:
### Module 2: ...
- Learning outcomes:
- Assignment:
### Module 3: ...
- Learning outcomes:
- Assignment:
### Module 4: ...
- Learning outcomes:
- Assignment:
### Module 5: ...
- Learning outcomes:
- Assignment:
## Assessment Plan
| Assessment | Description | Weight |
## Certification Criteria
## Marketing Strategy
- Step 1:
- Step 2:
- Step 3:
- Step 4:
- Step 5:

Make the course beginner-friendly and practical.
'''
response = call_response(prompt, model="gpt-5.4-mini")
print(response.output_text.strip())


## Course Overview
**Mastering Prompt Engineering for AI Applications** is a beginner-friendly, hands-on course designed to help learners write effective prompts for AI tools and models. The course focuses on practical techniques for improving output quality, reducing errors, and adapting prompts for real-world use cases such as writing, research, coding, automation, and customer support.

### Module 1: Introduction to Prompt Engineering
- Learning outcomes:
  - Understand what prompt engineering is and why it matters
  - Identify common types of AI applications that benefit from better prompting
  - Learn the basic structure of a strong prompt
- Assignment:
  - Write 5 simple prompts for different tasks and compare the outputs

### Module 2: Core Prompting Techniques
- Learning outcomes:
  - Use clarity, context, constraints, and examples to improve prompts
  - Learn how to ask AI for specific formats and tones
  - Recognize and fix vague or ineffective prompts
- Assignment:
  - Rewri

####